# 🛠️ Banking Credit Risk & Fraud Detection Analytics

## Notebook 05 — Feature Engineering

### Objective

The objective of this notebook is to transform the cleaned Credit Risk and Fraud Detection datasets into machine-learning-ready datasets.

Feature engineering involves creating, transforming, encoding, and selecting variables so that machine-learning algorithms can use the available information effectively.

### Main Activities

This notebook will cover:

- Loading cleaned datasets
- Separating features and target
- Identifying numerical and categorical features
- Handling missing values
- Encoding categorical variables
- Scaling numerical features
- Creating meaningful derived features
- Detecting potential data leakage
- Preparing train/test datasets
- Handling class imbalance
- Building reusable preprocessing pipelines
- Saving model-ready datasets and preprocessing objects

### Production Principle

All transformations must be reproducible.

The transformations learned from training data must also be applied consistently to validation, test, and future production data.

Therefore, scikit-learn pipelines will be used wherever appropriate.

## 1. Import Required Libraries

The following libraries will be used for data manipulation, feature engineering, preprocessing, and machine-learning pipeline construction.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

print("Feature engineering libraries imported successfully!")

Feature engineering libraries imported successfully!


## 2. Load Cleaned Datasets

The cleaned datasets produced during the previous stages will be loaded for feature engineering.

The Credit Risk dataset will be prepared for default prediction, while the Fraud Detection dataset will be prepared for fraud classification.

In [3]:
credit_df = pd.read_csv(
    "../data/processed/credit_risk_cleaned.csv"
)

fraud_df = pd.read_csv(
    "../data/processed/fraud_cleaned.csv"
)

print("Credit Risk Shape:", credit_df.shape)
print("Fraud Detection Shape:", fraud_df.shape)

Credit Risk Shape: (32416, 13)
Fraud Detection Shape: (283726, 31)


## 3. Data Leakage

Data leakage occurs when information that would not be available at prediction time is unintentionally used to train a machine-learning model.

Leakage can cause unrealistically high validation or test performance.

### Example

If we are predicting whether a loan applicant will default at the time of loan approval, information that becomes available only after the loan is issued should not be used as a predictor.

### General Rule

Every feature must answer:

> "Would this information genuinely be available at the time the prediction is made?"

For the Credit Risk dataset, `loan_status` is the target and must never be included as an input feature.

For the Fraud Detection dataset, `Class` is the target and must never be included as an input feature.

In [5]:
credit_target = "loan_status"
fraud_target = "Class"

print(
    "Credit Risk target:",
    credit_target
)

print(
    "Fraud Detection target:",
    fraud_target
)

Credit Risk target: loan_status
Fraud Detection target: Class


In [6]:
credit_features = credit_df.drop(
    columns=[credit_target]
)

fraud_features = fraud_df.drop(
    columns=[fraud_target]
)

print(
    "Credit feature shape:",
    credit_features.shape
)

print(
    "Fraud feature shape:",
    fraud_features.shape
)

Credit feature shape: (32416, 12)
Fraud feature shape: (283726, 30)


## 4. Separating Features and Target

Machine-learning problems are commonly represented using:

### X — Features

The independent/input variables used by the model.

### y — Target

The dependent/output variable the model is trying to predict.

For Credit Risk:

`X` → applicant and loan characteristics

`y` → `loan_status`

For Fraud Detection:

`X` → transaction characteristics

`y` → `Class`

In [8]:
X_credit = credit_df.drop(
    columns=["loan_status"]
)

y_credit = credit_df["loan_status"]

X_fraud = fraud_df.drop(
    columns=["Class"]
)

y_fraud = fraud_df["Class"]

print("Credit X:", X_credit.shape)
print("Credit y:", y_credit.shape)

print("Fraud X:", X_fraud.shape)
print("Fraud y:", y_fraud.shape)

Credit X: (32416, 12)
Credit y: (32416,)
Fraud X: (283726, 30)
Fraud y: (283726,)


## 5. Identify Numerical and Categorical Features

Machine-learning algorithms require different preprocessing strategies depending on the type of feature.

### Numerical Features

Examples:

- Income
- Age
- Loan amount
- Interest rate
- Employment length

These may require:

- Missing-value imputation
- Scaling
- Transformation

### Categorical Features

Examples:

- Home ownership
- Loan intent
- Loan grade
- Previous default indicator

These require conversion into numerical representations before most machine-learning algorithms can use them.

One-hot encoding will initially be used for nominal categorical variables.

In [9]:
credit_numeric_features = X_credit.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

credit_categorical_features = X_credit.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(credit_numeric_features)

print("\nCategorical features:")
print(credit_categorical_features)

Numerical features:
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'person_income_log']

Categorical features:
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']


## 6. Train-Test Split

The dataset will be divided into training and testing subsets.

The training set is used to:

- Learn preprocessing parameters
- Train the model

The test set is reserved for final unbiased evaluation.

### Stratification

Because default cases are less frequent than non-default cases, stratification will be used to preserve approximately the same target-class proportions in both datasets.

The test set will contain 20% of the observations.

In [10]:
X_credit_train, X_credit_test, y_credit_train, y_credit_test = (
    train_test_split(
        X_credit,
        y_credit,
        test_size=0.20,
        random_state=42,
        stratify=y_credit
    )
)

print("Training set:", X_credit_train.shape)
print("Testing set :", X_credit_test.shape)

print("\nTraining target distribution:")
print(y_credit_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_credit_test.value_counts(normalize=True))

Training set: (25932, 12)
Testing set : (6484, 12)

Training target distribution:
loan_status
0    0.781313
1    0.218687
Name: proportion, dtype: float64

Testing target distribution:
loan_status
0    0.781308
1    0.218692
Name: proportion, dtype: float64


## 7. Credit Risk Preprocessing Pipeline

A preprocessing pipeline will be created so that all transformations are learned from the training data and applied consistently to unseen data.

### Numerical Pipeline

1. Median imputation for missing values
2. Standard scaling

### Categorical Pipeline

1. Most-frequent-value imputation
2. One-hot encoding

### Column Transformer

The numerical and categorical pipelines will be combined using `ColumnTransformer`.

This structure allows the same preprocessing logic to be reused during model training, validation, testing, and deployment.

In [11]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [12]:
credit_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            credit_numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            credit_categorical_features
        )
    ]
)

print("Credit preprocessing pipeline created successfully!")

Credit preprocessing pipeline created successfully!


In [13]:
credit_preprocessor

,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [14]:
print("Training set:", X_credit_train.shape)
print("Testing set :", X_credit_test.shape)

Training set: (25932, 12)
Testing set : (6484, 12)


## 8. Business-Oriented Feature Engineering

Feature engineering will be performed using domain knowledge from lending and credit-risk analysis.

The objective is to create features that provide additional information about:

- Borrower financial capacity
- Loan burden
- Employment stability
- Credit history relative to applicant age

New features will be evaluated for redundancy and predictive usefulness before being included in the final model.

### Important Principle

A derived feature should not automatically replace its original variables.

Feature selection will be performed later using statistical analysis, model-based evaluation, multicollinearity diagnostics, and business reasoning.

## 8.1 Loan-to-Income Ratio

The loan-to-income ratio represents the requested loan amount relative to the applicant's annual income.

A higher ratio indicates that the requested loan represents a larger financial obligation relative to the applicant's income.

The dataset already contains `loan_percent_income`, which captures a similar concept. Therefore, the newly calculated ratio will primarily be used as a validation and feature-engineering exercise rather than automatically added alongside the existing feature.

In [15]:
credit_df["loan_to_income_ratio"] = (
    credit_df["loan_amnt"]
    / credit_df["person_income"]
)

credit_df[
    [
        "loan_amnt",
        "person_income",
        "loan_to_income_ratio",
        "loan_percent_income"
    ]
].head()

,loan_amnt,person_income,loan_to_income_ratio,loan_percent_income
0,35000,59000,0.593220,0.59
1,1000,9600,0.104167,0.10
2,5500,9600,0.572917,0.57
3,35000,65500,0.534351,0.53
4,35000,54400,0.643382,0.55


## 8.2 Employment-to-Age Ratio

Employment length alone does not account for applicant age.

A ratio between employment length and age can provide an approximate measure of the proportion of the applicant's life represented by the recorded employment history.

This is an exploratory feature and will be evaluated carefully because employment history and age may contain overlapping information.

In [16]:
credit_df["employment_age_ratio"] = (
    credit_df["person_emp_length"]
    / credit_df["person_age"]
)

credit_df[
    [
        "person_age",
        "person_emp_length",
        "employment_age_ratio"
    ]
].head()

,person_age,person_emp_length,employment_age_ratio
0,22.0,4.0,0.181818
1,21.0,5.0,0.238095
2,25.0,1.0,0.040000
3,23.0,4.0,0.173913
4,24.0,8.0,0.333333


## 8.3 Credit History-to-Age Ratio

`cb_person_cred_hist_length` and `person_age` are strongly correlated.

A derived ratio can provide additional context by representing credit-history length relative to applicant age.

This feature is exploratory and will later be evaluated against the original variables for redundancy and predictive usefulness.

In [17]:
credit_df["credit_history_age_ratio"] = (
    credit_df["cb_person_cred_hist_length"]
    / credit_df["person_age"]
)

credit_df[
    [
        "person_age",
        "cb_person_cred_hist_length",
        "credit_history_age_ratio"
    ]
].head()

,person_age,cb_person_cred_hist_length,credit_history_age_ratio
0,22.0,3,0.136364
1,21.0,2,0.095238
2,25.0,3,0.120000
3,23.0,2,0.086957
4,24.0,4,0.166667


## 8.4 Validate Engineered Features

Before using engineered variables in modeling, their distributions and missing values should be checked.

This prevents unexpected values or invalid calculations from silently entering the modeling pipeline.

In [18]:
engineered_features = [
    "loan_to_income_ratio",
    "employment_age_ratio",
    "credit_history_age_ratio"
]

credit_df[engineered_features].describe()

,loan_to_income_ratio,employment_age_ratio,credit_history_age_ratio
count,32416.000000,32416.000000,32416.000000
mean,0.170599,0.173430,0.194924
std,0.107081,0.133527,0.093676
min,0.000789,0.000000,0.076923
25%,0.089711,0.066667,0.125000
50%,0.148148,0.148148,0.173913
75%,0.229167,0.272727,0.261905
max,0.830000,0.716981,0.961538


In [19]:
credit_df[engineered_features].isnull().sum()

loan_to_income_ratio        0
employment_age_ratio        0
credit_history_age_ratio    0
dtype: int64

In [20]:
np.isinf(
    credit_df[engineered_features]
).sum()

loan_to_income_ratio        0
employment_age_ratio        0
credit_history_age_ratio    0
dtype: int64

## 8.5 Engineered Feature Redundancy Analysis

Engineered variables may contain information that is already represented by existing variables.

Correlation analysis will be used as an initial diagnostic for redundancy.

Highly correlated features will not automatically be removed. Their usefulness will ultimately be evaluated through model performance, multicollinearity diagnostics, and business interpretability.

In [21]:
feature_check_columns = [
    "loan_amnt",
    "person_income",
    "loan_percent_income",
    "person_age",
    "person_emp_length",
    "cb_person_cred_hist_length",
    "loan_to_income_ratio",
    "employment_age_ratio",
    "credit_history_age_ratio"
]

engineered_corr = credit_df[
    feature_check_columns
].corr()

engineered_corr

,loan_amnt,person_income,loan_percent_income,person_age,person_emp_length,cb_person_cred_hist_length,loan_to_income_ratio,employment_age_ratio,credit_history_age_ratio
loan_amnt,1.000000,0.265947,0.572824,0.051443,0.111769,0.041865,0.577228,0.103906,0.038294
person_income,0.265947,1.000000,-0.254472,0.118019,0.136640,0.117614,-0.253188,0.110786,0.111447
loan_percent_income,0.572824,-0.254472,1.000000,-0.041518,-0.058488,-0.031457,0.998939,-0.050758,-0.026004
person_age,0.051443,0.118019,-0.041518,1.000000,0.170589,0.877886,-0.041383,-0.057526,0.698003
person_emp_length,0.111769,0.136640,-0.058488,0.170589,1.000000,0.148142,-0.057273,0.952586,0.134583
cb_person_cred_hist_length,0.041865,0.117614,-0.031457,0.877886,0.148142,1.000000,-0.031425,-0.054424,0.938777
loan_to_income_ratio,0.577228,-0.253188,0.998939,-0.041383,-0.057273,-0.031425,1.000000,-0.049688,-0.026043
employment_age_ratio,0.103906,0.110786,-0.050758,-0.057526,0.952586,-0.054424,-0.049688,1.000000,-0.033945
credit_history_age_ratio,0.038294,0.111447,-0.026004,0.698003,0.134583,0.938777,-0.026043,-0.033945,1.000000


In [22]:
credit_df[engineered_features].describe()

,loan_to_income_ratio,employment_age_ratio,credit_history_age_ratio
count,32416.000000,32416.000000,32416.000000
mean,0.170599,0.173430,0.194924
std,0.107081,0.133527,0.093676
min,0.000789,0.000000,0.076923
25%,0.089711,0.066667,0.125000
50%,0.148148,0.148148,0.173913
75%,0.229167,0.272727,0.261905
max,0.830000,0.716981,0.961538


In [23]:
np.isinf(
    credit_df[engineered_features]
).sum()

loan_to_income_ratio        0
employment_age_ratio        0
credit_history_age_ratio    0
dtype: int64

## 9. Final Credit Risk Feature Set

Feature engineering produced several candidate variables.

After reviewing business meaning and potential redundancy, the initial modeling feature set will prioritize:

### Applicant Features

- `person_age`
- `person_income`
- `person_income_log`
- `person_emp_length`
- `person_home_ownership`

### Loan Features

- `loan_intent`
- `loan_grade`
- `loan_amnt`
- `loan_int_rate`
- `loan_percent_income`

### Credit History Features

- `cb_person_default_on_file`
- `cb_person_cred_hist_length`

### Engineered Feature

- `employment_age_ratio`
- `credit_history_age_ratio`

### Excluded Candidate

`loan_to_income_ratio` will not be included because `loan_percent_income` already represents a very similar concept.

The final feature set is an initial modeling configuration and will be evaluated empirically during model development.

In [24]:
credit_model_features = [
    "person_age",
    "person_income",
    "person_income_log",
    "person_emp_length",
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_default_on_file",
    "cb_person_cred_hist_length",
    "employment_age_ratio",
    "credit_history_age_ratio"
]

X_credit = credit_df[
    credit_model_features
].copy()

y_credit = credit_df[
    "loan_status"
].copy()

print("Final Credit X shape:", X_credit.shape)
print("Credit y shape:", y_credit.shape)

Final Credit X shape: (32416, 14)
Credit y shape: (32416,)


In [25]:
X_credit_train, X_credit_test, y_credit_train, y_credit_test = (
    train_test_split(
        X_credit,
        y_credit,
        test_size=0.20,
        random_state=42,
        stratify=y_credit
    )
)

print("Training features:", X_credit_train.shape)
print("Testing features :", X_credit_test.shape)

print("\nTraining target:")
print(y_credit_train.value_counts(normalize=True))

print("\nTesting target:")
print(y_credit_test.value_counts(normalize=True))

Training features: (25932, 14)
Testing features : (6484, 14)

Training target:
loan_status
0    0.781313
1    0.218687
Name: proportion, dtype: float64

Testing target:
loan_status
0    0.781308
1    0.218692
Name: proportion, dtype: float64


In [26]:
credit_numeric_features = X_credit_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

credit_categorical_features = X_credit_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(credit_numeric_features)

print("\nCategorical features:")
print(credit_categorical_features)

Numerical features:
['person_age', 'person_income', 'person_income_log', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'employment_age_ratio', 'credit_history_age_ratio']

Categorical features:
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']


In [27]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

credit_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            credit_numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            credit_categorical_features
        )
    ]
)

print("Final Credit Risk preprocessing pipeline created.")

Final Credit Risk preprocessing pipeline created.


## 10. Fit Preprocessing Only on Training Data

The preprocessing pipeline must learn parameters only from the training dataset.

Examples include:

- Median values used for imputation
- Mean and standard deviation used by StandardScaler
- Categories learned by OneHotEncoder

The test dataset must remain completely unseen during this learning process.

This prevents preprocessing-related data leakage.

In [28]:
X_credit_train_processed = credit_preprocessor.fit_transform(
    X_credit_train
)

X_credit_test_processed = credit_preprocessor.transform(
    X_credit_test
)

print(
    "Processed training shape:",
    X_credit_train_processed.shape
)

print(
    "Processed testing shape:",
    X_credit_test_processed.shape
)

Processed training shape: (25932, 29)
Processed testing shape: (6484, 29)


In [30]:
print(
    "Training matrix contains NaN:",
    np.isnan(
        X_credit_train_processed.toarray()
        if hasattr(
            X_credit_train_processed,
            "toarray"
        )
        else X_credit_train_processed
    ).any()
)

print(
    "Testing matrix contains NaN:",
    np.isnan(
        X_credit_test_processed.toarray()
        if hasattr(
            X_credit_test_processed,
            "toarray"
        )
        else X_credit_test_processed
    ).any()
)

Training matrix contains NaN: False
Testing matrix contains NaN: False


## 11. Save Preprocessing Pipeline

The fitted preprocessing pipeline will be saved so that the exact same transformations can be reused during model inference and deployment.

This is essential for production consistency.

The pipeline will later be loaded by the API and applied to incoming applicant data before generating predictions.

In [31]:
import joblib

joblib.dump(
    credit_preprocessor,
    "../models/credit_preprocessor.pkl"
)

print(
    "Credit preprocessing pipeline saved successfully."
)

Credit preprocessing pipeline saved successfully.
